<h1>Rediscovering particles using real LHC (Large Hadron Collider) data from CERN</h1>
<p>The purpose of this project is to familiarise myself with one of the key data formats that CERN uses, and to use my existing python knowledge, paired with my own research, to look into using the CMS (compact muon solenoid) data to rediscover particles.</p>
<p>Firstly, we need to import the python libraries necessary to handle CERN's root data into something manageable by python</p>

In [6]:
import uproot
import awkward as ak

my_file = uproot.open("../data/cern_dataset_2012.root")
tree = my_file["Events"]
print(f"number of entries: {tree.num_entries}\n")
tree.show()
print()

number of entries: 61540413

name                 | typename                 | interpretation                
---------------------+--------------------------+-------------------------------
nMuon                | uint32_t                 | AsDtype('>u4')
Muon_pt              | float[]                  | AsJagged(AsDtype('>f4'))
Muon_eta             | float[]                  | AsJagged(AsDtype('>f4'))
Muon_phi             | float[]                  | AsJagged(AsDtype('>f4'))
Muon_mass            | float[]                  | AsJagged(AsDtype('>f4'))
Muon_charge          | int32_t[]                | AsJagged(AsDtype('>i4'))



<p>Now let's take a look at certain specific events to see what we can find...</p>

In [8]:
ev = tree.arrays(entry_stop=10)  # Load only the first 10 events
print("nMuon: ", ev.nMuon)  # Muons in each event
print("Muon_pt: ",ev.Muon_pt)  # transverse momentum of the muons
print("Muon_charge: ", ev.Muon_charge)  # +1 or -1 for each muon, representing positive or neg. charge

nMuon:  [2, 2, 1, 4, 4, 3, 2, 2, 2, 2]
Muon_pt:  [[10.8, 15.7], [10.5, 16.3], [3.28], ..., [...], [10.2, 14.2], [11.5, 3.47]]
Muon_charge:  [[-1, -1], [1, -1], [1], [1, 1, 1, 1], ..., [...], [1, -1], [-1, 1], [-1, 1]]


<h2>Introducing numpy to do a deeper analysis of the data...</h2>
<p>Now lets count the muons and check the charges! (of the first 100,000 events)</p>
<p>We will take a look at which events have 2, 3, 4... muons and so on</p>

In [9]:
ev = tree.arrays(["nMuon", "Muon_charge"], entry_stop=100_000)  # we will cap at 100k

# How many events have 2,3,4 etc. muons

# Bring in numpy for better analysis
import numpy as np
values, counts = np.unique(ak.to_numpy(ev.nMuon), return_counts=True)
# Return in 'pairs' so to speak, like a dictionary
print({int(v): int(c) for v, c in zip(values, counts)})

{0: 3210, 1: 13447, 2: 48976, 3: 21726, 4: 8021, 5: 2752, 6: 1066, 7: 414, 8: 198, 9: 80, 10: 56, 11: 22, 12: 11, 13: 5, 14: 8, 15: 4, 16: 1, 18: 1, 23: 1, 39: 1}


<p>As we can see above, there were <b>13,447</b> collisions with one muon, and <b>48,976</b> with two, also being the most common.</p>
<h4>We will now filter, of the two-muon events, which ones have muons of opposing charges??</h4>
<i>sidenote: make sure all cells are ran chronologically :)</i>

In [18]:
two_muon = ev[ev.nMuon == 2]
opposite = ak.sum(two_muon.Muon_charge, axis=1) == 0
print(str(round(ak.mean(opposite) * 100))+"%")  # returns as a decimal 0 - 1 then multiply by 100 so 0.759... ≈ 76%

76%


<h4>Why do we mainly care about two muons & opposing charges?</h4>
<p>With exactly two muons there's only one possible pair, so we don't have to guess which muons belong together. Many of these pairs are still unrelated background; the real particles show up as peaks above it.</p>
<p>And with regards to opposing charges -> the particles we are to discover in this notebook are neutrally charged, therefore following basic conservation of charge laws, the charges MUST cancel each other out. If they happen to be of the same charge, we consider them background particles and disregard them</p>

<h1>Analysing the data</h1>
<p>we are to stop here at stage 11, continue later on ...</p>

In [1]:
"""
Dimuon invariant mass spectrum from CMS Open Data (2012, 8 TeV)
Reads muon pairs from a CMS file, reconstructs their invariant mass, plots the full spectrum and fits the Z-boson peak.
"""
import argparse
from pathlib import Path

import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
import uproot
from scipy.optimize import curve_fit
from scipy.special import voigt_profile

DATA_URL = (
"https://opendata.cern.ch/eos/opendata/cms/derived-data/"
"AOD2NanoAODOutreachTool/Run2012BC_DoubleMuParked_Muons.root"
)
BRANCHES = ["nMuon","Muon_pt","Muon_eta","Muon_phi","Muon_mass","Muon_charge"]
LOCAL_FILE = "data/cern_dataset_2012.root"
Z_MASS_PDG = 91.1876 # GeV, Particle Data Group

# Known resonances to label on the plot: (name, mass in GeV)
RESONANCES = [
    (r"$\eta$", 0.548), (r"$\rho,\omega$", 0.78), (r"$\phi$",1.02),
    (r"J/$\psi$", 3.097), (r"$\psi$'", 3.686), (r"$\Upsilon$", 9.46), ("Z", 91.19)
]

<p>The above code does not 'action' anything, rather sets some important variables and parameters</p>
<p>The invariant mass function is next</p>
<p>The invariant mass function takes two particles measured by a detector and works out the mass of the particle that has decayed into them.</p>
<p>First, it converts each particle's detector coordinates (pt, eta, phi) into ordinary x, y, z momentum. Then it finds each particle's energy using E^2 = p^2 + m^2</p>
<p>Since energy and momentum are conserved, adding the two particles' values gives the parent particle's total energy and momentum. Finally m^2 = E^2 - p^2 gives the parents mass.</p>
<p>This value is "invariant" because it comes out the same however fast the parent was moving</p>

In [5]:
# mass invariant function
def invariant_mass(pt, eta, phi, mass):
    # invariant mass of two particles, each argument has shape (n_events, 2)
    px = pt * np.cos(phi)
    py = pt * np.sin(phi)
    pz = pt * np.sinh(eta)
    energy = np.sqrt(px**2 + py**2 + pz**2 + mass**2)

    # Sum the two muons' four-momenta, then m^2 = E^2 - |p|^2
    e, x, y, z = (a.sum(axis=1) for a in (energy, px, py, pz))
    return np.sqrt(np.maximum(e**2 - x**2 - y**2 - z**2, 0))

<q>Why vectorised NumPy operations?</q>
<p>Numpy runs these operations compiled, in C, over the whole array, if we used python instead, it <b>could be up to 100 times slower</b>, because we are interpreting each operation rather than using a compiled library!</p>

<h2>Reading the file in chunks and selecting events</h2>
<ul>
    <li>tree.iterate(...) will read 1 million events at a time (due to step size), therefore memory usage stays plateaued even on massive files.</li>
    <li>The selection line builds a boolean mask: nMuon == 2 and the charges sum to zero e.g. (-1, +1). Indexing with this mask keeps <b>only these important events</b></li>
    <li>Because every selected event now has exactly two muons, ak.to_numpy can turn the jagged arrays into regular (n,2) NumPy arrays.</li>
    <li>entry_stop allows us to stop at N events, good for when testing.</li>
</ul>

In [6]:
def process(source, max_events=None, step_size=1_000_000):
    # read the file in chunks and return dimuon masses of selected events
    masses, n_total = [], 0
    tree = uproot.open(source)["Events"]
    for chunk in tree.iterate(BRANCHES, step_size=step_size, entry_stop=max_events):
        n_total += len(chunk)
        # Selection criteria, two muons with opposite charge...
        selected = chunk[(chunk.nMuon == 2) & (ak.sum(chunk.Muon_charge, axis=1) == 0)]
        cols = [ak.to_numpy(selected[b]) for b in ("Muon_pt", "Muon_eta", "Muon_phi", "Muon_mass")]
        masses.append(invariant_mass(*cols))
        print(f"Processed {n_total:,} events")

    masses = np.concatenate(masses)
    print(f"Selected {len(masses):,} of {n_total:,} events")
    return masses

<h2>Plotting the spectrum</h2>
<p>Both axes need to be <b>logarithmic</b> because the peaks span 0.5 to 100 GeV and the counts span four orders of magnitude. np.logspace makes bins that are equally wide on the log scale, so narrow low-mass peaks and the broad Z peak are both visible. The loop places each particle's label just above the histogram at its known mass.</p>

In [9]:
def plot_spectrum(masses, out):
    bins = np.logspace(np.log10(0.25), np.log10(300), 300)
    fig, ax = plt.subplots(figsize=(10,6))
    counts, _, _ = ax.hist(masses, bins=bins, histtype="step", color="black", lw=1)
    ax.set(xscale="log", yscale="log", xlabel=r"Dimuon mass $m_{\mu\mu}$ [GeV]", ylabel="Events per bin", title="CMS Open Data 2012, $\\sqrt{s}$ = 8 TeV")

    for name, m in RESONANCES:
        height = counts[np.searchsorted(bins, m) - 1]
        ax.annotate(name, (m, height * 1.6), ha="center", fontsize=11)

    fig.tight_layout()
    fig.savefig(out, dpi=150)
    print(f"Saved {out}")
    

<h2>Fitting the Z-peak</h2>
<p>We now turn the picture into an actual measurement</p>
<ul>
    <li>An unstable particle's mass follows a Breit-Wigner shape, with natural width Γ. The detector measures momentum imperfectly, which smears the peak like a gaussian width σ. The combination is a Voigt profile. The a + b(x - mu) term is a straight-line background under the peak.</li>
    <li><b>The errors:</b> The count in each bin is a Poisson variable, so the uncertainty is sqrt(N). Passing sigma=errors makes the fit weight each bin correctly.</li>
    <li><b>The result:</b> The curve_fit returns us the best fit parameters and their covariance matrix; the square roots of its diagonal are the parameter uncertainties.</li>
    <li>p0 is the starting guess. Fits need sensible starting values to converge!!</li>
</ul>

In [11]:
def fit_z_peak(masses, out, lo=70.0, hi=110.0, nbins=80):
    # fitting the Z peak with a voigt profile + linear background...
    counts, edges = np.histogram(masses, bins=nbins, range=(lo, hi))
    centres = 0.5 * (edges[1:] + edges[:-1])
    width = edges[1] - edges[0]

    def model(x, n, mu, sigma, gamma, a, b):
        return n * width * voigt_profile(x - mu, sigma, gamma / 2) + a + b * (x - mu)

    errors = np.sqrt(np.maximum(counts, 1))  # poisson uncertainties.
    p0 = [counts.sum(), 91.0, 1.5, 2.5, 1.0, 0.0]
    popt, pcov = curve_fit(model, centres, counts, p0=p0, sigma=errors, absolute_sigma=True)
    perr = np.sqrt(np.diag(pcov))
    mu, mu_err, gamma, gamma_err = popt[1], perr[1], popt[3], perr[3]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.errorbar(centres, counts, yerr=errors, fmt="o", ms=3, color="black", label="Data")
    x = np.linspace(lo, hi, 500)
    ax.plot(x, model(x, *popt), color="tab:red",
        label=f"Fit: $m_Z$ = {mu:.2f} $\\pm$ {mu_err:.2f} GeV")
    ax.set(xlabel=r"$m_{\mu\mu}$ [GeV]", ylabel=f"Events / {width:.1f} GeV", title="Z boson peak")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out, dpi=150)

    diff = 100 * (mu - Z_MASS_PDG) / Z_MASS_PDG
    print(f"Fitted Z mass: {mu:.3f} +/- {mu_err:.3f} GeV (stat.)")
    print(f"PDG Z mass:   {Z_MASS_PDG} GeV -> difference {diff:+.2f}%")
    print(f"Saved {out}")

In [ ]:
masses = process("../data/cern_dataset_2012.root", max_events=1_000_000)  # remove max_events for the full run
plot_spectrum(masses, "../plots/dimuon_spectrum.png")
fit_z_peak(masses, "../plots/z_peak_fit.png")